In [1]:
import os
import json
import datetime
import random
import zipfile
import numpy as np
import pandas as pd
import torch
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    confusion_matrix,
    classification_report,
)
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import Dataset
from datasets import load_dataset
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    get_cosine_schedule_with_warmup,
)
from transformers import logging as hf_logging

hf_logging.set_verbosity_error()

import warnings

warnings.filterwarnings("ignore")

os.environ["PYTHONWARNINGS"] = "ignore"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seeds = [42, 1337, 2026]

data_dir = r"C:\Users\X\Desktop\X\ALTA\alta2026_public_data"
root_dir = r"C:\Users\X\Desktop\X\ALTA"
test_csv = r"C:\Users\X\Desktop\X\ALTA\sample_test\answer.csv"

model_name = "microsoft/deberta-v3-large"
max_length = 256
grad_accum_steps = 4

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

task_configs = {
    "sentiment": {
        "epochs": 8,
        "batch_size": 4,
        "lr": 1.0e-5,
        "patience": 3,
        "label_smoothing": 0.05,
        "layer_decay": 0.9,
        "classifier_dropout": 0.1,
        "weight_key": "variety",
        "extra_multiplier": None,
        "train_source_filter": None,
    },
    "sarcasm": {
        "epochs": 12,
        "batch_size": 4,
        "lr": 8.0e-6,
        "patience": 4,
        "label_smoothing": 0.1,
        "layer_decay": 0.85,
        "classifier_dropout": 0.2,
        "weight_key": "group_key",
        "extra_multiplier": None,
        "train_source_filter": "reddit",
    },
}

external_sarcasm_epochs = 2
external_sarcasm_lr = 2.0e-5
external_sarcasm_batch_size = 16
external_sarcasm_max_examples = 15000
external_sarcasm_valid_examples = 1000
external_checkpoint_dir_name = "sarcasm_sarc_pretrained"

run_id = (
    datetime.datetime.now().strftime("%Y%m%d-%H%M%S") + f"_{model_name.split('/')[-1]}_ensemble"
)

run_dir = os.path.join(root_dir, "runs", run_id)
work_dir = os.path.join(run_dir, "tf_runs")
model_out_dir = os.path.join(run_dir, "final_models")
metrics_dir = os.path.join(run_dir, "metrics")
out = os.path.join(run_dir, "answer.csv")
submission_zip = os.path.join(run_dir, "submission.zip")

for d in (run_dir, work_dir, model_out_dir, metrics_dir):
    os.makedirs(d, exist_ok=True)

run_config = {
    "run_id": run_id,
    "timestamp": datetime.datetime.now().isoformat(),
    "model_name": model_name,
    "max_length": max_length,
    "grad_accum_steps": grad_accum_steps,
    "seeds": seeds,
    "task_configs": task_configs,
    "data_dir": data_dir,
    "use_bf16": use_bf16,
    "use_fp16": use_fp16,
}

with open(os.path.join(run_dir, "run_config.json"), "w") as f:
    json.dump(run_config, f, indent=2)


class TextDataset(Dataset):
    def __init__(self, encodings, labels=None, sample_weights=None):
        self.encodings = encodings
        self.labels = labels
        self.sample_weights = sample_weights

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}

        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx])

        if self.sample_weights is not None:
            item["sample_weight"] = torch.tensor(self.sample_weights[idx], dtype=torch.float)

        return item


class WeightedCollator(DataCollatorWithPadding):
    def __call__(self, features):
        weights = [f.pop("sample_weight") for f in features if "sample_weight" in f]

        batch = super().__call__(features)

        if weights:
            batch["sample_weight"] = torch.stack(weights)

        return batch


def make_weighted_trainer(label_smoothing):
    class WeightedTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.pop("labels")
            weight = inputs.pop("sample_weight", None)

            outputs = model(**inputs)
            logits = outputs.logits

            per_sample = torch.nn.functional.cross_entropy(
                logits, labels, reduction="none", label_smoothing=label_smoothing
            )

            if weight is not None:
                weight = weight.to(logits.device)
                loss = (per_sample * weight).sum() / weight.sum()
            else:
                loss = per_sample.mean()

            return (loss, outputs) if return_outputs else loss

    return WeightedTrainer


def get_device():
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def grouped_class_weights(labels, group, extra_multiplier=None):
    labels = np.array(labels)
    group = np.array(group)
    weights = np.ones(len(labels), dtype=np.float32)

    for g in np.unique(group):
        mask = group == g

        cw = compute_class_weight(class_weight="balanced", classes=np.array([0, 1]), y=labels[mask])

        for c in (0, 1):
            w = cw[c]

            if extra_multiplier and (g, c) in extra_multiplier:
                w *= extra_multiplier[(g, c)]

            weights[mask & (labels == c)] = w

    return weights


def make_compute_metrics(variety_array):
    variety_array = np.asarray(variety_array)
    varieties = sorted(set(variety_array))

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)

        result = {"macro_f1": f1_score(labels, preds, average="macro")}

        per_variety = []

        for v in varieties:
            mask = variety_array == v

            f1 = f1_score(labels[mask], preds[mask], average="macro")

            result[f"macro_f1_{v}"] = f1
            per_variety.append(f1)

        result["weak_macro_f1"] = min(per_variety)

        return result

    return compute_metrics


def build_optimizer(model, base_lr, layer_decay, weight_decay=0.01):
    no_decay = ["bias", "LayerNorm.weight", "layer_norm.weight"]

    num_layers = model.config.num_hidden_layers
    groups = {}

    def layer_id_of(name):
        if "embeddings" in name:
            return 0

        if ".layer." in name:
            return int(name.split(".layer.")[1].split(".")[0]) + 1

        return num_layers + 1

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue

        layer_id = layer_id_of(name)

        scale = layer_decay ** (num_layers + 1 - layer_id)

        decay = 0.0 if any(nd in name for nd in no_decay) else weight_decay

        key = (layer_id, decay)

        if key not in groups:
            groups[key] = {"params": [], "lr": base_lr * scale, "weight_decay": decay}

        groups[key]["params"].append(param)

    return torch.optim.AdamW(list(groups.values()), lr=base_lr)


def pretrain_sarcasm_external(tokenizer, device, base_model_name):
    external_dir = os.path.join(model_out_dir, external_checkpoint_dir_name)

    if os.path.exists(os.path.join(external_dir, "config.json")):
        print(f"\nExternal sarcasm checkpoint already exists at {external_dir}, skipping pretrain.")
        return external_dir

    print("\nLoading external sarcasm dataset (SARC / daniel2588/sarcasm)...")

    ext = load_dataset("daniel2588/sarcasm")
    ext_df = ext["train"].to_pandas()

    text_col = None
    for candidate in ("comment", "text", "body"):
        if candidate in ext_df.columns:
            text_col = candidate
            break

    label_col = None
    for candidate in ("label", "is_sarcastic", "sarcasm"):
        if candidate in ext_df.columns:
            label_col = candidate
            break

    if text_col is None or label_col is None:
        raise ValueError(
            f"Could not find expected text/label columns in daniel2588/sarcasm. "
            f"Columns found: {list(ext_df.columns)}. "
            f"Update text_col/label_col detection above to match."
        )

    print(f"  Using text column {text_col!r}, label column {label_col!r}")
    print(f"  Raw rows: {len(ext_df)}")

    ext_df = ext_df[[text_col, label_col]].dropna()
    ext_df[label_col] = ext_df[label_col].astype(int)
    ext_df = ext_df[ext_df[label_col].isin([0, 1])]

    print(f"  After cleaning: {len(ext_df)}")
    print(f"  Label balance: {ext_df[label_col].value_counts().to_dict()}")

    rng = np.random.RandomState(seeds[0])

    total_needed = external_sarcasm_max_examples + external_sarcasm_valid_examples
    if len(ext_df) > total_needed:
        ext_df_0 = ext_df[ext_df[label_col] == 0]
        ext_df_1 = ext_df[ext_df[label_col] == 1]

        half = total_needed // 2
        n0 = min(half, len(ext_df_0))
        n1 = min(half, len(ext_df_1))

        ext_df_0 = ext_df_0.sample(n=n0, random_state=seeds[0])
        ext_df_1 = ext_df_1.sample(n=n1, random_state=seeds[0])

        ext_df = pd.concat([ext_df_0, ext_df_1]).sample(frac=1.0, random_state=seeds[0])

    ext_df = ext_df.reset_index(drop=True)

    n_valid = min(external_sarcasm_valid_examples, max(1, len(ext_df) // 10))
    ext_valid_df = ext_df.iloc[:n_valid]
    ext_train_df = ext_df.iloc[n_valid:]

    print(f"  Using {len(ext_train_df)} train / {len(ext_valid_df)} valid examples for pretraining")

    ext_train_texts = ext_train_df[text_col].astype(str).tolist()
    ext_train_labels = ext_train_df[label_col].tolist()

    ext_valid_texts = ext_valid_df[text_col].astype(str).tolist()
    ext_valid_labels = ext_valid_df[label_col].tolist()

    ext_train_enc = tokenizer(ext_train_texts, truncation=True, max_length=max_length)
    ext_valid_enc = tokenizer(ext_valid_texts, truncation=True, max_length=max_length)

    ext_train_ds = TextDataset(ext_train_enc, ext_train_labels)
    ext_valid_ds = TextDataset(ext_valid_enc, ext_valid_labels)

    config = AutoConfig.from_pretrained(base_model_name, num_labels=2)

    model = AutoModelForSequenceClassification.from_pretrained(
        base_model_name,
        config=config,
        torch_dtype=torch.float32,
    )

    model.to(device)
    model.gradient_checkpointing_enable()

    collator = DataCollatorWithPadding(tokenizer)

    steps_per_epoch = max(
        1, int(np.ceil(len(ext_train_ds) / (external_sarcasm_batch_size * grad_accum_steps)))
    )
    total_steps = steps_per_epoch * external_sarcasm_epochs
    warmup_steps = int(0.1 * total_steps)

    optimizer = torch.optim.AdamW(model.parameters(), lr=external_sarcasm_lr, weight_decay=0.01)

    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    def ext_compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return {"macro_f1": f1_score(labels, preds, average="macro")}

    training_args = TrainingArguments(
        output_dir=os.path.join(work_dir, "external_sarcasm_pretrain"),
        num_train_epochs=external_sarcasm_epochs,
        per_device_train_batch_size=external_sarcasm_batch_size,
        per_device_eval_batch_size=external_sarcasm_batch_size * 2,
        gradient_accumulation_steps=grad_accum_steps,
        learning_rate=external_sarcasm_lr,
        warmup_ratio=0.1,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        logging_steps=25,
        fp16=use_fp16,
        bf16=use_bf16,
        gradient_checkpointing=True,
        eval_accumulation_steps=1,
        max_grad_norm=1.0,
        seed=seeds[0],
        data_seed=seeds[0],
        report_to=[],
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=ext_train_ds,
        eval_dataset=ext_valid_ds,
        data_collator=collator,
        compute_metrics=ext_compute_metrics,
        optimizers=(optimizer, scheduler),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    print("Pretraining on external SARC sarcasm data before ALTA sarcasm fine-tuning...")
    trainer.train()

    metrics = trainer.evaluate()
    print(f"External SARC pretrain final macro_f1: {metrics.get('eval_macro_f1'):.4f}")

    os.makedirs(external_dir, exist_ok=True)
    trainer.save_model(external_dir)
    tokenizer.save_pretrained(external_dir)

    del model, trainer, optimizer, scheduler
    torch.cuda.empty_cache()

    return external_dir


def train_one_task(
    task, cfg, train_df, valid_df, test_df, tokenizer, device, seed, base_model_name
):
    train_source_filter = cfg.get("train_source_filter")

    if train_source_filter is not None:
        task_train_df = train_df[train_df["source"] == train_source_filter].reset_index(drop=True)
        print(
            f"  Filtering train set to source={train_source_filter!r}: "
            f"{len(task_train_df)} / {len(train_df)} rows"
        )
    else:
        task_train_df = train_df

    train_enc = tokenizer(list(task_train_df["text"]), truncation=True, max_length=max_length)

    valid_enc = tokenizer(list(valid_df["text"]), truncation=True, max_length=max_length)

    test_enc = tokenizer(list(test_df["text"]), truncation=True, max_length=max_length)

    train_labels = task_train_df[task].tolist()

    sample_weights = grouped_class_weights(
        train_labels,
        task_train_df[cfg["weight_key"]].tolist(),
        extra_multiplier=cfg["extra_multiplier"],
    )

    train_ds = TextDataset(train_enc, train_labels, sample_weights)

    valid_ds = TextDataset(valid_enc, valid_df[task].tolist())

    test_ds = TextDataset(test_enc)

    config = AutoConfig.from_pretrained(base_model_name, num_labels=2)

    config.classifier_dropout = cfg["classifier_dropout"]
    config.hidden_dropout_prob = cfg["classifier_dropout"]

    model = AutoModelForSequenceClassification.from_pretrained(
        base_model_name,
        config=config,
        torch_dtype=torch.float32,
    )

    model.to(device)

    model.gradient_checkpointing_enable()

    assert all(p.dtype == torch.float32 for p in model.parameters())

    collator = WeightedCollator(tokenizer)

    optimizer = build_optimizer(model, cfg["lr"], cfg["layer_decay"])

    effective_batch_size = cfg["batch_size"] * grad_accum_steps

    steps_per_epoch = max(1, int(np.ceil(len(train_ds) / effective_batch_size)))

    total_steps = steps_per_epoch * cfg["epochs"]
    warmup_steps = int(0.1 * total_steps)

    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    training_args = TrainingArguments(
        output_dir=os.path.join(work_dir, f"{task}_seed{seed}"),
        num_train_epochs=cfg["epochs"],
        per_device_train_batch_size=cfg["batch_size"],
        per_device_eval_batch_size=cfg["batch_size"] * 2,
        gradient_accumulation_steps=grad_accum_steps,
        learning_rate=cfg["lr"],
        warmup_ratio=0.1,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="weak_macro_f1",
        greater_is_better=True,
        logging_steps=25,
        fp16=use_fp16,
        bf16=use_bf16,
        gradient_checkpointing=True,
        eval_accumulation_steps=1,
        max_grad_norm=1.0,
        seed=seed,
        data_seed=seed,
        report_to=[],
        remove_unused_columns=False,
    )

    WeightedTrainer = make_weighted_trainer(cfg["label_smoothing"])

    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=valid_ds,
        data_collator=collator,
        compute_metrics=make_compute_metrics(valid_df["variety"].to_numpy()),
        optimizers=(optimizer, scheduler),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=cfg["patience"])],
    )

    trainer.train()

    save_path = os.path.join(model_out_dir, f"{task}_seed{seed}")

    os.makedirs(save_path, exist_ok=True)

    trainer.save_model(save_path)
    tokenizer.save_pretrained(save_path)

    with open(os.path.join(metrics_dir, f"{task}_seed{seed}_trainer_log_history.json"), "w") as f:
        json.dump(trainer.state.log_history, f, indent=2)

    valid_preds = trainer.predict(valid_ds)
    test_preds = trainer.predict(test_ds)

    valid_probs = torch.softmax(torch.tensor(valid_preds.predictions), dim=-1).numpy()
    test_probs = torch.softmax(torch.tensor(test_preds.predictions), dim=-1).numpy()

    del model, trainer
    torch.cuda.empty_cache()

    return valid_probs, test_probs


def plot_confusion_matrix(cm, class_names, title, save_path):
    fig, ax = plt.subplots(figsize=(4, 4))

    im = ax.imshow(cm, cmap="Blues")

    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

    ax.set_xticks(range(len(class_names)))
    ax.set_yticks(range(len(class_names)))

    ax.set_xticklabels(class_names)
    ax.set_yticklabels(class_names)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j,
                i,
                str(cm[i, j]),
                ha="center",
                va="center",
                color=("white" if cm[i, j] > cm.max() / 2 else "black"),
            )

    fig.colorbar(im, ax=ax)

    fig.tight_layout()

    fig.savefig(save_path, dpi=150)

    plt.close(fig)


def compute_all_metrics(gold_df, run_df):
    all_metrics = {}
    official_scores = {}
    class_names = ["0", "1"]

    for task in ["sentiment", "sarcasm"]:
        all_metrics[task] = {}

        for dialect in ["en-AU", "en-UK"]:
            gold = gold_df[gold_df["variety"] == dialect][task]

            pred = run_df[run_df["variety"] == dialect][task]

            macro_f1 = f1_score(gold, pred, average="macro")

            macro_precision = precision_score(gold, pred, average="macro", zero_division=0)

            macro_recall = recall_score(gold, pred, average="macro", zero_division=0)

            acc = accuracy_score(gold, pred)

            report = classification_report(gold, pred, output_dict=True, zero_division=0)

            cm = confusion_matrix(gold, pred, labels=[0, 1])

            all_metrics[task][dialect] = {
                "accuracy": acc,
                "macro_f1": macro_f1,
                "macro_precision": macro_precision,
                "macro_recall": macro_recall,
                "classification_report": report,
                "confusion_matrix": cm.tolist(),
            }

            official_scores[f"{task}-{dialect}"] = macro_f1

            cm_df = pd.DataFrame(
                cm,
                index=[f"true_{c}" for c in class_names],
                columns=[f"pred_{c}" for c in class_names],
            )

            cm_df.to_csv(os.path.join(metrics_dir, f"cm_{task}_{dialect}.csv"))

            plot_confusion_matrix(
                cm,
                class_names,
                title=f"{task} — {dialect}",
                save_path=os.path.join(metrics_dir, f"cm_{task}_{dialect}.png"),
            )

        gold_all = gold_df[task]
        pred_all = run_df[task]

        cm_all = confusion_matrix(gold_all, pred_all, labels=[0, 1])

        all_metrics[task]["overall"] = {
            "accuracy": accuracy_score(gold_all, pred_all),
            "macro_f1": f1_score(gold_all, pred_all, average="macro"),
            "macro_precision": precision_score(
                gold_all, pred_all, average="macro", zero_division=0
            ),
            "macro_recall": recall_score(gold_all, pred_all, average="macro", zero_division=0),
            "classification_report": classification_report(
                gold_all, pred_all, output_dict=True, zero_division=0
            ),
            "confusion_matrix": cm_all.tolist(),
        }

        cm_df_all = pd.DataFrame(
            cm_all,
            index=[f"true_{c}" for c in class_names],
            columns=[f"pred_{c}" for c in class_names],
        )

        cm_df_all.to_csv(os.path.join(metrics_dir, f"cm_{task}_overall.csv"))

        plot_confusion_matrix(
            cm_all,
            class_names,
            title=f"{task} — overall",
            save_path=os.path.join(metrics_dir, f"cm_{task}_overall.png"),
        )

    final = (
        min(official_scores["sentiment-en-AU"], official_scores["sentiment-en-UK"])
        + min(official_scores["sarcasm-en-AU"], official_scores["sarcasm-en-UK"])
    ) / 2

    all_metrics["official_weak_macro_f1_per_task_dialect"] = official_scores

    all_metrics["official_final_score"] = final

    return all_metrics, final


device = get_device()

print(f"Run ID: {run_id}")
print(f"Outputs will be written under: {run_dir}")
print(f"Using device: {device}")
print(f"Precision: bf16={use_bf16} " f"fp16={use_fp16}")
print(f"Model: {model_name}")
print(f"Gradient accumulation: {grad_accum_steps}")
print(f"Seeds: {seeds}")

train_df = pd.read_csv(os.path.join(data_dir, "train.csv"))

valid_df = pd.read_csv(os.path.join(data_dir, "valid.csv"))

test_df = pd.read_csv(test_csv)

for d in (train_df, valid_df, test_df):
    d["group_key"] = d["source"] + "_" + d["variety"]
    d["text"] = "[" + d["source"] + "/" + d["variety"] + "] " + d["text"]

tokenizer = AutoTokenizer.from_pretrained(model_name)

set_seed(seeds[0])
sarcasm_base_model = pretrain_sarcasm_external(tokenizer, device, model_name)

task_base_model = {
    "sentiment": model_name,
    "sarcasm": sarcasm_base_model,
}

valid_probs_by_seed = {task: [] for task in ["sentiment", "sarcasm"]}
test_probs_by_seed = {task: [] for task in ["sentiment", "sarcasm"]}

for s in seeds:
    print(f"\nSeed {s} :")
    set_seed(s)

    for task in ["sentiment", "sarcasm"]:
        print(f"\nFinetuning for task: {task} (seed {s})")

        vp, tp = train_one_task(
            task,
            task_configs[task],
            train_df,
            valid_df,
            test_df,
            tokenizer,
            device,
            s,
            task_base_model[task],
        )

        valid_probs_by_seed[task].append(vp)
        test_probs_by_seed[task].append(tp)

        np.save(os.path.join(metrics_dir, f"{task}_seed{s}_valid_probs.npy"), vp)
        np.save(os.path.join(metrics_dir, f"{task}_seed{s}_test_probs.npy"), tp)

valid_preds = {}
test_preds = {}

for task in ["sentiment", "sarcasm"]:
    valid_avg_probs = np.mean(np.stack(valid_probs_by_seed[task], axis=0), axis=0)
    test_avg_probs = np.mean(np.stack(test_probs_by_seed[task], axis=0), axis=0)

    valid_preds[task] = np.argmax(valid_avg_probs, axis=-1)
    test_preds[task] = np.argmax(test_avg_probs, axis=-1)

    np.save(os.path.join(metrics_dir, f"{task}_ensemble_valid_probs.npy"), valid_avg_probs)
    np.save(os.path.join(metrics_dir, f"{task}_ensemble_test_probs.npy"), test_avg_probs)

valid_answer_df = valid_df[["source", "variety", "text"]].copy()

valid_answer_df["sentiment"] = valid_preds["sentiment"]

valid_answer_df["sarcasm"] = valid_preds["sarcasm"]

test_answer_df = test_df[["source", "variety", "text"]].copy()

test_answer_df["sentiment"] = test_preds["sentiment"]

test_answer_df["sarcasm"] = test_preds["sarcasm"]

test_answer_df.to_csv(out, index=False)

print(f"\nWrote test predictions to {out}")

with zipfile.ZipFile(submission_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(out, arcname="answer.csv")

print(f"Wrote submission zip to {submission_zip}")

all_metrics, final = compute_all_metrics(valid_df, valid_answer_df)

with open(os.path.join(metrics_dir, "all_metrics.json"), "w") as f:
    json.dump(all_metrics, f, indent=2)

print("\nOfficial style scores " "(weak macro F1) on the public valid set:")

for k, v in all_metrics["official_weak_macro_f1_per_task_dialect"].items():
    print(f"  {k}: {v:.4f}")

print(f"final score: {final:.4f}")

print("\nAll metrics, confusion matrices, " f"and models saved under:\n  {run_dir}")

Run ID: 20260925-172915_deberta-v3-large_ensemble
Outputs will be written under: C:\Users\X\Desktop\X\ALTA\runs\20260925-172915_deberta-v3-large_ensemble
Using device: cuda
Precision: bf16=True fp16=False
Model: microsoft/deberta-v3-large
Gradient accumulation: 4
Seeds: [42, 1337, 2026]

Loading external sarcasm dataset (SARC / daniel2588/sarcasm)...
  Using text column 'comment', label column 'label'
  Raw rows: 1010826
  After cleaning: 1010771
  Label balance: {0: 505403, 1: 505368}
  Using 15000 train / 1000 valid examples for pretraining


Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

Pretraining on external SARC sarcasm data before ALTA sarcasm fine-tuning...
{'loss': '2.789', 'grad_norm': '10.06', 'learning_rate': '1.021e-05', 'epoch': '0.1066'}
{'loss': '2.657', 'grad_norm': '16.07', 'learning_rate': '2e-05', 'epoch': '0.2132'}
{'loss': '2.564', 'grad_norm': '30.56', 'learning_rate': '1.98e-05', 'epoch': '0.3198'}
{'loss': '2.399', 'grad_norm': '21.75', 'learning_rate': '1.926e-05', 'epoch': '0.4264'}
{'loss': '2.282', 'grad_norm': '34.73', 'learning_rate': '1.841e-05', 'epoch': '0.533'}
{'loss': '2.297', 'grad_norm': '19.49', 'learning_rate': '1.727e-05', 'epoch': '0.6397'}
{'loss': '2.143', 'grad_norm': '15.56', 'learning_rate': '1.587e-05', 'epoch': '0.7463'}
{'loss': '2.144', 'grad_norm': '11.23', 'learning_rate': '1.428e-05', 'epoch': '0.8529'}
{'loss': '2.236', 'grad_norm': '9.587', 'learning_rate': '1.253e-05', 'epoch': '0.9595'}
{'eval_loss': '0.5297', 'eval_macro_f1': '0.7398', 'eval_runtime': '2.782', 'eval_samples_per_second': '359.5', 'eval_steps_per_

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '1.949', 'grad_norm': '20.64', 'learning_rate': '1.07e-05', 'epoch': '1.064'}
{'loss': '1.836', 'grad_norm': '13.55', 'learning_rate': '8.851e-06', 'epoch': '1.171'}
{'loss': '1.718', 'grad_norm': '18.38', 'learning_rate': '7.037e-06', 'epoch': '1.277'}
{'loss': '1.751', 'grad_norm': '17.95', 'learning_rate': '5.325e-06', 'epoch': '1.384'}
{'loss': '1.87', 'grad_norm': '27.14', 'learning_rate': '3.773e-06', 'epoch': '1.49'}
{'loss': '1.687', 'grad_norm': '15.38', 'learning_rate': '2.436e-06', 'epoch': '1.597'}
{'loss': '1.788', 'grad_norm': '16.84', 'learning_rate': '1.358e-06', 'epoch': '1.704'}
{'loss': '1.757', 'grad_norm': '20.03', 'learning_rate': '5.779e-07', 'epoch': '1.81'}
{'loss': '1.687', 'grad_norm': '21.22', 'learning_rate': '1.214e-07', 'epoch': '1.917'}
{'eval_loss': '0.5146', 'eval_macro_f1': '0.76', 'eval_runtime': '4.871', 'eval_samples_per_second': '205.3', 'eval_steps_per_second': '6.57', 'epoch': '2'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (117 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '3224', 'train_samples_per_second': '10.3', 'train_steps_per_second': '0.644', 'train_loss': '2.067', 'epoch': '9'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Wrote test predictions to C:\Users\X\Desktop\X\ALTA\runs\20260925-172915_deberta-v3-large_ensemble\answer.csv
Wrote submission zip to C:\Users\X\Desktop\X\ALTA\runs\20260925-172915_deberta-v3-large_ensemble\submission.zip

Official style scores (weak macro F1) on the public valid set:
  sentiment-en-AU: 0.9375
  sentiment-en-UK: 0.9636
  sarcasm-en-AU: 0.7578
  sarcasm-en-UK: 0.8260
final score: 0.8477

All metrics, confusion matrices, and models saved under:
  C:\Users\X\Desktop\X\ALTA\runs\20260925-172915_deberta-v3-large_ensemble
